# Part 5: Toàn Bộ Mô Hình Học Máy (Data Splitting, Supervised & Unsupervised Learning)
**Thành viên phụ trách:** Thành viên 5 | **Nhánh Git:** `feature/ml-modeling` (hoặc `feature/ml-insight-analysis`)
---
Notebook tổng hợp toàn diện các giai đoạn Machine Learning của dự án FlightRisk VN:
1. **Chiến lược Phân chia Dữ liệu:** Chronological Train/Test Split (80% Train, 20% Test, mốc cắt ngày `2026-10-30`, chống Lookahead Bias).
2. **Học Có Giám Sát (Supervised Learning):**
   - Huấn luyện 3 mô hình phân loại: Baseline Logistic Regression, Balanced Random Forest, HistGradientBoosting Classifier.
   - Đánh giá toàn diện (Accuracy, Precision, Recall, Macro F1, ROC-AUC, Confusion Matrix).
   - Huấn luyện mô hình Hồi quy dự báo số phút trễ (`delay_min`).
   - Trích xuất Top Feature Importance.
3. **Học Không Giám Sát (Unsupervised Learning) & Khai phá Tri thức:**
   - Phân cụm K-Means 4 cụm rủi ro vận hành (Silhouette Score, Inertia, đặc tả hồ sơ cụm).
   - Giảm chiều dữ liệu PCA (2D & 3D, Explained Variance Ratio).
   - Phát hiện bất thường Isolation Forest (385 chuyến bay dị biệt, contamination=3%).
   - Khai phá luật kết hợp FP-Growth (Association Rules Mining).


### 1. Phân Chia Dữ Liệu Theo Chuỗi Thời Gian (Chronological Split - Chống Lookahead Bias)


In [ ]:
from modeling.train import split_by_date, prepare_chronological_train_test
import pandas as pd

X_train, X_test, y_train, y_test, delay_train, delay_test = prepare_chronological_train_test()
print('\n=== THỐNG KÊ PHÂN CHIA TẬP DỮ LIỆU ===')
print(f'Tập Huấn Luyện (Train): {len(X_train):,} dòng ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)')
print(f'Tập Kiểm Thử (Test)   : {len(X_test):,} dòng ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)')
print(f'Số đặc trưng đầu vào  : {X_train.shape[1]} đặc trưng')
print('Phân bố nhãn Train     :\n', y_train.value_counts(normalize=True).round(3))
print('Phân bố nhãn Test      :\n', y_test.value_counts(normalize=True).round(3))


### 2. Huấn Luyện & Đánh Giá Mô Hình Học Có Giám Sát (Supervised Learning)


In [ ]:
from modeling.train import run_supervised_training
from config import BASE_DIR
import json

run_supervised_training()

with open(BASE_DIR / 'models' / 'supervised_metrics.json', encoding='utf-8') as f:
    sup_metrics = json.load(f)

print(f"\n=== MÔ HÌNH PHÂN LOẠI TỐT NHẤT: {sup_metrics['classification']['best_classifier_name']} ===")
for model_name, metrics in sup_metrics['classification']['models'].items():
    print(f'\nModel: {model_name}')
    print(f"  Accuracy       : {metrics['accuracy']:.4f}")
    print(f"  Macro F1-Score : {metrics['f1_macro']:.4f}")
    print(f"  Weighted F1    : {metrics['f1_weighted']:.4f}")
    print(f"  ROC-AUC (ovr)  : {metrics.get('roc_auc_ovr', 'N/A')}")

print('\n=== KẾT QUẢ MÔ HÌNH HỒI QUY DỰ BÁO PHÚT TRỄ ===')
print(f"  MAE  : {sup_metrics['regression']['MAE_minutes']} phút")
print(f"  RMSE : {sup_metrics['regression']['RMSE_minutes']} phút")
print(f"  R2   : {sup_metrics['regression']['R2_Score']}")

print('\n=== TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT ===')
for rank, item in enumerate(sup_metrics['feature_importances'][:10], 1):
    print(f"  {rank:2d}. {item['feature']:<30}: {item['importance']*100:.2f}%")


### 3. Huấn Luyện & Khám Phá Mô Hình Học Không Giám Sát (Unsupervised Learning)


In [ ]:
from modeling.unsupervised import run_unsupervised_pipeline
from config import BASE_DIR
import json, pandas as pd

run_unsupervised_pipeline()

with open(BASE_DIR / 'models' / 'cluster_profiles.json', encoding='utf-8') as f:
    profiles = json.load(f)

with open(BASE_DIR / 'models' / 'unsupervised_metrics.json', encoding='utf-8') as f:
    unsup_metrics = json.load(f)

print('\n=== HỒ SƠ 4 CỤM RỦI RO K-MEANS ===')
profiles_df = pd.DataFrame(list(profiles.values()))
display(profiles_df[['cluster_id', 'name', 'flight_count', 'percentage', 'delay_rate_pct', 'cancellation_rate_pct', 'dominant_airline']])

print('\n=== THỐNG KÊ GIẢM CHIỀU PCA & DỊ THƯỜNG ISOLATION FOREST ===')
print(f"PCA 2 thành phần giải thích phương sai : {unsup_metrics['pca']['explained_variance_ratio_2d']}")
print(f"Tổng phương sai tích lũy (2D)          : {unsup_metrics['pca']['cumulative_variance_2d']*100:.2f}%")
print(f"Số chuyến bay bất thường phát hiện     : {unsup_metrics['isolation_forest']['anomaly_count']:,} chuyến ({unsup_metrics['isolation_forest']['anomaly_pct']}%)")
